# Проверка baseline

Импорт необходимых для работы библиотек

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [14]:
import sys
import os
sys.path.append(os.path.abspath(".."))
from evaluate_sku import split_history, compute_wape, compute_mase_per_sku

In [12]:
train = pd.read_csv('../data/train.csv')

Строим прогнозы для окна 

In [16]:
cutoff = "2021-08"
history, actual = split_history(train, cutoff)

In [20]:
display(history)

,sku,unit,2019-01,2019-02,2019-03,2019-04,2019-05,2019-06,2019-07,2019-08,...,2020-11,2020-12,2021-01,2021-02,2021-03,2021-04,2021-05,2021-06,2021-07,2021-08
0,0,ШТ,128.0,125.0,179.0,199.0,204.0,246.0,172.0,235.0,...,175.0,271.0,335.0,379.0,250.0,317.0,172.0,193.0,173.0,121.0
1,1,ШТ,65.0,64.0,54.0,91.0,74.0,91.0,138.0,115.0,...,85.0,80.0,47.0,68.0,65.0,72.0,82.0,85.0,69.0,75.0
2,2,ШТ,157.0,205.0,210.0,205.0,265.0,248.0,292.0,268.0,...,350.0,327.0,333.0,334.0,380.0,409.0,309.0,388.0,438.0,404.0
3,3,ШТ,2450.0,2280.0,2144.0,2283.0,2177.0,2888.0,3066.0,2288.0,...,1642.0,1625.0,2537.0,2806.0,2085.0,1984.0,2086.0,2871.0,2204.0,1821.0
4,4,ШТ,119.0,154.0,154.0,270.0,513.0,718.0,720.0,292.0,...,392.0,599.0,221.0,222.0,258.0,289.0,662.0,759.0,616.0,365.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1651,1651,ШТ,536.0,920.0,1503.0,3268.0,3878.0,6611.0,8629.0,6499.0,...,1551.0,1506.0,1857.0,1482.0,1850.0,2930.0,5484.0,12128.0,6177.0,6953.0
1652,1652,ПК,138.0,247.0,255.0,2033.0,1268.0,476.0,851.0,643.0,...,198.0,258.0,110.0,255.0,244.0,2431.0,1096.0,496.0,523.0,586.0
1653,1653,ШТ,708.0,730.0,828.0,959.0,918.0,908.0,1049.0,1094.0,...,1165.0,1526.0,1074.0,941.0,1353.0,1167.0,1035.0,1025.0,1158.0,1258.0
1654,1654,ШТ,594.0,654.0,820.0,740.0,697.0,690.0,1006.0,703.0,...,934.0,968.0,667.0,888.0,1226.0,1133.0,802.0,893.0,860.0,973.0


In [19]:
display(actual)

,id,actual
0,0_2021-09,318.0
1,1_2021-09,100.0
2,2_2021-09,293.0
3,3_2021-09,2187.0
4,4_2021-09,321.0
...,...,...
4963,1651_2021-11,1672.0
4964,1652_2021-11,160.0
4965,1653_2021-11,2129.0
4966,1654_2021-11,1171.0


In [26]:
# 1. Берём 'id' и вытаскиваем номер товара 'sku'
sub_last = actual[['id']].copy()
sub_last['sku'] = sub_last['id'].str.split('_').str[0].astype(int)

# 2. Подтягиваем продажи последнего месяца из history и называем столбец 'prediction'
sub_last = sub_last.merge(history[['sku', cutoff]], on='sku')
sub_last = sub_last.rename(columns={cutoff: 'prediction'})

# 3. Сопоставляем факт и прогноз строго по общему 'id'
eval_df = actual.merge(sub_last[['id', 'prediction']], on='id')

# 4. Теперь считаем WAPE
wape_last = compute_wape(eval_df['actual'].to_numpy(), eval_df['prediction'].to_numpy())
print(f"WAPE (Последний месяц): {wape_last:.4f}")

WAPE (Последний месяц): 0.3804


Результат слишком высокий

In [27]:
from evaluate_sku import build_evaluation

cutoffs = ['2020-11', '2021-02', '2021-05', '2021-08']
month_cols = [c for c in train.columns if c.startswith('20')]

wape_results = []

for cutoff in cutoffs:
    history, actuals = split_history(train, cutoff=cutoff)
    cutoff_idx = month_cols.index(cutoff)
    target_months = month_cols[cutoff_idx + 1 : cutoff_idx + 4]
    
    # 1. Бейзлайн: 12 месяцев назад (сезонный)
    seasonal_months = [month_cols[month_cols.index(m) - 12] for m in target_months]
    seasonal_melt = history.melt(id_vars='sku', value_vars=seasonal_months, var_name='s_month', value_name='prediction')
    month_map = dict(zip(seasonal_months, target_months))
    seasonal_melt['target_month'] = seasonal_melt['s_month'].map(month_map)
    seasonal_melt['id'] = seasonal_melt['sku'].astype(str) + '_' + seasonal_melt['target_month']
    
    eval_seasonal = build_evaluation(actuals, seasonal_melt[['id', 'prediction']])
    wape_seasonal = compute_wape(eval_seasonal['actual'].to_numpy(), eval_seasonal['prediction'].to_numpy())
    
    # 2. Бейзлайн: Последний месяц
    sub_last = actuals[['id']].copy()
    sub_last['sku'] = sub_last['id'].str.split('_').str[0].astype(int)
    sub_last = sub_last.merge(history[['sku', cutoff]], on='sku').rename(columns={cutoff: 'prediction'})
    
    eval_last = build_evaluation(actuals, sub_last[['id', 'prediction']])
    wape_last = compute_wape(eval_last['actual'].to_numpy(), eval_last['prediction'].to_numpy())
    
    # 3. Бейзлайн: Среднее за последние 12 месяцев
    last_12 = month_cols[cutoff_idx - 11 : cutoff_idx + 1]
    mean_df = pd.DataFrame({'sku': history['sku'], 'prediction': history[last_12].mean(axis=1)})
    sub_mean = actuals[['id']].copy()
    sub_mean['sku'] = sub_mean['id'].str.split('_').str[0].astype(int)
    sub_mean = sub_mean.merge(mean_df, on='sku')
    
    eval_mean = build_evaluation(actuals, sub_mean[['id', 'prediction']])
    wape_mean = compute_wape(eval_mean['actual'].to_numpy(), eval_mean['prediction'].to_numpy())
    
    wape_results.append({
        'Период проверки': f"{target_months[0]} — {target_months[-1]}",
        '12 мес. назад': round(wape_seasonal, 3),
        'Последний месяц': round(wape_last, 3),
        'Среднее за 12 мес.': round(wape_mean, 3)
    })

df_wape = pd.DataFrame(wape_results)
df_wape

,Период проверки,12 мес. назад,Последний месяц,Среднее за 12 мес.
0,2020-12 — 2021-02,0.321,0.358,0.555
1,2021-03 — 2021-05,0.388,0.454,0.354
2,2021-06 — 2021-08,0.435,0.289,0.322
3,2021-09 — 2021-11,0.311,0.380,0.275
